# Evaluation Notebook

This file performs evaluation of all algorithms/variants, stores results in a Zarr file, and generates plots.

## Configuration

In [1]:
# Set the default model directory.
# model_dir = "/data/group/mas/qmas/results/patrolling_zoo_v0/mappo/diffuser-patrolling-obs200/wandb/run-20260316_170952-7dqf5om8/files"
# model_dir = "/data/group/mas/qmas/results/patrolling_zoo_v0/rmappo/onemodule-patrolling-obs200-uqEnsemble3-pygNS-commsRL-injectionAppend-predObs/wandb/run-20260414_172146-w98za9r3/files"
# model_dir = "/data/group/mas/qmas/results/patrolling_zoo_v0/rmappo/diffuser-patrolling-random-obs100-obsPredUQ-commsRL0.1-rwdNew-64x3/wandb/run-20260422_123712-04y9696v/files"
# model_dir = "/data/group/mas/qmas/results/patrolling_zoo_v0/rmappo/diffuser-patrolling-random-obs200-coords/wandb/run-20260423_132222-n53k4u8z/files/episode_15"
model_dir = "/data/group/mas/qmas/results/patrolling_zoo_v0/rmappo/diffuser-patrolling-obs200-coords/wandb/run-20260423_154056-cgt3hjvy/files/episode_15"

# Set the output file.
# output_file = "output.zarr"

# Set the output file automatically.
model_dir_parts = model_dir.split("/")
run_name = model_dir_parts[next(i for i, part in enumerate(model_dir_parts) if part.startswith("run-"))]
output_file = "./patrolling_" + run_name + ".zarr"

## Setup

In [2]:
%reload_ext autoreload
%autoreload 2
from onpolicy.scripts.eval.eval_pettingzoo import main

import os
import yaml
import time
from copy import deepcopy
from multiprocessing import Process
os.environ["WANDB__SERVICE_WAIT"] = "300"

In [3]:
def convert_args(argsdict):
    # Convert arguments to the appropriate format.
    arglist = []
    def add_arg(arglist, key, value):
        if value == None:
            return
        if type(value) == bool:
            if value:
                arglist.append(f"--{key}")
            else:
                arglist.append(f"--no-{key}")
        elif type(value) == list and len(value) > 0:
            arglist.append(f"--{key}")
            arglist.extend([str(v) for v in value])
        else:
            arglist.append(f"--{key}")
            arglist.append(str(value))
    for key, value in argsdict.items():
        if type(value) == dict and "value" in value and not key.startswith("_"):
            add_arg(arglist, key, value["value"])
        else:
            add_arg(arglist, key, value)
    print("Converted arguments:", arglist)
    return arglist


def wrap_arg_val(val):
    return {"value": val}


def load_args(model_dir):
    # Load arguments from the config file.
    config_file = os.path.join(model_dir, "config.yaml")
    args = yaml.load(open(config_file), Loader=yaml.FullLoader)

    # Set required eval-specific arguments. Do not change these!
    args["use_wandb"] = wrap_arg_val(False)
    args["use_render"] = wrap_arg_val(False)
    args["use_eval"] = wrap_arg_val(True)
    args["model_dir"] = wrap_arg_val(model_dir)
    args["eval_output_file"] = wrap_arg_val(output_file)


    # Set default arguments for evaluation. Feel free to change these!
    args["cuda"] = wrap_arg_val(True)
    args["cuda_idx"] = wrap_arg_val(7)
    args["cuda_idx_predictor"] = wrap_arg_val(7)
    args["eval_episodes"] = wrap_arg_val(3)
    args["episode_length"] = wrap_arg_val(200)
    args["seed"] = wrap_arg_val(1)

    args["observation_mask"] = wrap_arg_val(True)

    return args

# Load arguments for the default model.
args = load_args(model_dir)

# Evaluation

In [4]:
# Check if the output file already exists. If so, skip this cell.
if os.path.exists(output_file):
    print("Output file already exists. Skipping evaluation.")

else:
    processes = []

    def start_process(args):
        p = Process(target=main, args=[convert_args(args)])
        processes.append(p)

    # obs_probabilities = np.arange(0.0, 1.0, 0.01).tolist()
    # obs_probabilities = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
    # comms_probabilities = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
    comms_probabilities = [0.0]

    for comms_prob in comms_probabilities:
        args = load_args(model_dir)

        # series_name = args["experiment_name"]['value']
        series_name = model_dir.split("/")[-3]

        args["eval_series"] = wrap_arg_val(f"AR0 - {series_name}")
        # args["experiment_name"] = wrap_arg_val(f"comms_prob_{comms_prob:.2f}")
        # args["communication_probability"] = wrap_arg_val(comms_prob)
        args["diffusion_autoregression_steps"] = wrap_arg_val(0)
        start_process(args)
        
        argscopy = deepcopy(args)
        argscopy["eval_series"] = wrap_arg_val(f"AR1 - {series_name}")
        argscopy["diffusion_autoregression_steps"] = wrap_arg_val(1)
        start_process(argscopy)

        argscopy = deepcopy(args)
        argscopy["eval_series"] = wrap_arg_val(f"AR7 - {series_name}")
        argscopy["diffusion_autoregression_steps"] = wrap_arg_val(7)
        start_process(argscopy)

        # VARIANT 2: No prediction.
        argscopy = deepcopy(args)
        argscopy["eval_series"] = wrap_arg_val(f"no-prediction - {series_name}")
        argscopy["communication_probability"] = wrap_arg_val(comms_prob)
        argscopy["prediction_disable"] = wrap_arg_val(True)
        start_process(argscopy)


    # VARIANT 3: Load an entirely different model! (example)
    # argsnew = load_args("/data/group/mas/qmas/results/patrolling_zoo_v0/mappo/diffuser-patrolling-obs200/wandb/run-20250731_182446-4h2njfwx/files")
    # argsnew["experiment_name"] = wrap_arg_val("diffusion-prediction-obs200")
    # start_process(argsnew)


    # JOIN ALL PROCESSES
    # Up to MAX_PROCESSES processes can run simultaneously.
    MAX_PROCESSES = 4
    running_processes = []
    time_start = time.time()
    failed_processes = []
    
    while len(processes) > 0 or len(running_processes) > 0:
        # Start new processes if we have capacity.
        while len(running_processes) < MAX_PROCESSES and len(processes) > 0:
            p = processes.pop(0)
            p.start()
            running_processes.append(p)
        
        # Check for finished processes.
        for p in running_processes:
            if not p.is_alive():
                p.join()
                if p.exitcode != 0:
                    failed_processes.append(p)
        running_processes = [p for p in running_processes if p.is_alive()]
    
    time_end = time.time()
    print(f"Total evaluation time: {time_end - time_start:.2f} seconds")
    
    if failed_processes:
        raise RuntimeError(f"{len(failed_processes)} subprocess(es) failed with non-zero exit codes")
    
    print("All processes finished successfully!")

Converted arguments: ['--accept_ratio', '0.5', '--action_full_max_nodes', '40', '--action_method', 'neighbors_with_comm_boolean', '--action_neighbors_max_degree', '10', '--agent_speed', '40.0', '--algorithm_class', 'qmas.variant_two_modules.new.algorithm.QmasAlgorithm', '--algorithm_name', 'rmappo', '--alpha', '1.0', '--attrition_fixed_times', '[]', '--attrition_method', 'none', '--attrition_min_agents', '2', '--attrition_random_probability', '0.0', '--beta', '0.1', '--broadcast_uncertainty_threshold', '0.5', '--clip_param', '0.2', '--communication_model', 'none', '--communication_probability', '0.0', '--critic_hidden_size', '64', '--critic_lr', '0.0005', '--cuda', '--cuda_deterministic', '--cuda_idx', '7', '--cuda_idx_predictor', '7', '--data_chunk_length', '10', '--no-dec_actor', '--diffusion_autoregression_steps', '0', '--diffusion_epoch', '5', '--diffusion_model_type', 'dit1d', '--diffusion_steps', '20', '--no-encode_state', '--entropy_coef', '0.01', '--env_class', 'patrolling_zoo.

/home/anthony/data/dev/qmas_task_allocation/onpolicy/scripts/train/train_pettingzoo.py:158: UserWarning: Unknown arguments: ['--episode_max', '75', '--no-share_reward', '--no-torch_compile', '--no-use_gnn_critic', '--no-use_gnn_mlp_policy']
  warnings.warn(f"Unknown arguments: {unknown_args}")


u are choosing to use rmappo, we set use_recurrent_policy to be True
Pettingzoo environment patrolling_zoo.patrolling_zoo_v0.parallel_env has additional arguments.


/home/anthony/data/dev/qmas_task_allocation/onpolicy/scripts/train/train_pettingzoo.py:158: UserWarning: Unknown arguments: ['--episode_max', '75', '--no-share_reward', '--no-torch_compile', '--no-use_gnn_critic', '--no-use_gnn_mlp_policy']
  warnings.warn(f"Unknown arguments: {unknown_args}")


u are choosing to use rmappo, we set use_recurrent_policy to be TruePettingzoo environment patrolling_zoo.patrolling_zoo_v0.parallel_env has additional arguments.



...

u are choosing to use rmappo, we set use_recurrent_policy to be True
Pettingzoo environment patrolling_zoo.patrolling_zoo_v0.parallel_env has additional arguments.


/home/anthony/data/dev/qmas_task_allocation/onpolicy/scripts/train/train_pettingzoo.py:158: UserWarning: Unknown arguments: ['--episode_max', '75', '--no-share_reward', '--no-torch_compile', '--no-use_gnn_critic', '--no-use_gnn_mlp_policy']
  warnings.warn(f"Unknown arguments: {unknown_args}")


u are choosing to use rmappo, we set use_recurrent_policy to be True
Pettingzoo arguments validated: base
Pettingzoo environment patrolling_zoo.patrolling_zoo_v0.parallel_env has additional arguments validation function.
Pettingzoo arguments validated: patrolling_zoo.patrolling_zoo_v0.parallel_env
PettingzooEnv: Attempting to pass unpacked argparse namespace to environment.
PettingzooEnv: Passing the following arguments to the environment: {'max_cycles': -1, 'num_agents': 4, 'alpha': 1.0, 'beta': 0.1, 'reward_comms_penalty_weight': 0.1, 'graph_file': '../patrolling_zoo/graphs/milwaukee.graph', 'graph_random': False, 'graph_random_nodes': 40, 'reward_method_terminal': 'none', 'reward_interval': 1, 'agent_speed': 40.0, 'action_method': 'neighbors_with_comm_boolean', 'observe_method': 'pyg', 'observe_method_global': 'coordinates', 'observe_bitmap_size': 50, 'observation_radius': 200.0, 'attrition_method': 'none', 'attrition_fixed_times': ['[', ']'], 'attrition_random_probability': 0.0, 'a

/home/anthony/miniconda3/envs/qmas/lib/python3.10/site-packages/torchrl/__init__.py:48: UserWarning: failed to set start method to spawn, and current start method for mp is fork.
  warn(
/home/anthony/miniconda3/envs/qmas/lib/python3.10/site-packages/torchrl/__init__.py:48: UserWarning: failed to set start method to spawn, and current start method for mp is fork.
  warn(
/home/anthony/miniconda3/envs/qmas/lib/python3.10/site-packages/torchrl/__init__.py:48: UserWarning: failed to set start method to spawn, and current start method for mp is fork.
  warn(
/home/anthony/miniconda3/envs/qmas/lib/python3.10/site-packages/torchrl/__init__.py:48: UserWarning: failed to set start method to spawn, and current start method for mp is fork.
  warn(


R_Actor: Use GNN: True, Use CNN: False, Use MLP: False, Use RNN: True, Use Encoder: True
State encoder use_cnn: False, input_dim: 258, output_dim: 32
R_Actor: Use GNN: True, Use CNN: False, Use MLP: False, Use RNN: True, Use Encoder: True
State encoder use_cnn: False, input_dim: 258, output_dim: 32
Action output distributions:
  0: Categorical, action_dim: 1, available_action_dim: 2Action output distributions:

  1: Categorical, action_dim: 1, available_action_dim: 10  0: Categorical, action_dim: 1, available_action_dim: 2

  1: Categorical, action_dim: 1, available_action_dim: 10
R_Actor: Use GNN: True, Use CNN: False, Use MLP: False, Use RNN: True, Use Encoder: True
State encoder use_cnn: False, input_dim: 258, output_dim: 32
Action output distributions:
  0: Categorical, action_dim: 1, available_action_dim: 2
  1: Categorical, action_dim: 1, available_action_dim: 10
R_Actor: Use GNN: True, Use CNN: False, Use MLP: False, Use RNN: True, Use Encoder: True
State encoder use_cnn: False,

# Analysis

In [5]:
import zarr
output = zarr.open(output_file, mode='r')
output.tree()

Tree(nodes=(Node(disabled=True, name='/', nodes=(Node(disabled=True, name='AR0 - run-20260423_154056-cgt3hjvy'…

In [7]:
import plotly.graph_objects as go
import numpy as np

# Data for plotting
categories = {
    'prediction_error_mean': {
        'title': 'Prediction Error',
        'ylabel': 'Prediction Error (units distance)',
        'xlabel': 'Communication Probability',
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][experiment]['prediction_error_mean']) for experiment in output[series]]
                for series in output
        }
    },
    'prediction_trajectory_error_mean': {
        'title': 'Prediction Trajectory Error',
        'ylabel': 'Prediction Trajectory Error (units distance)',
        'xlabel': 'Communication Probability',
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][experiment]['prediction_trajectory_error_mean']) for experiment in output[series]]
                for series in output
        }
    },
    'avg_idleness': {
        'title': 'Time-Averaged Avg. Idleness',
        'ylabel': 'Avg. Node Idleness (s)',
        'xlabel': 'Communication Probability',
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][experiment]['avg_idleness']) for experiment in output[series]]
                for series in output
        }
    },
    'avg_idleness_final': {
        'title': 'Final Avg. Idleness',
        'ylabel': 'Avg. Node Idleness (s)',
        'xlabel': 'Communication Probability',
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][experiment]['avg_idleness'][:, -1]) for experiment in output[series]]
                for series in output
        }
    },
    'avg_idleness_over_time': {
        'title': 'Avg. Idleness vs. Time',
        'ylabel': 'Avg. Node Idleness (s)',
        'xlabel': 'Time Steps',
        # One trace per test/experiment: average over episodes, keep time dimension.
        'y': {
            series: {
                experiment: np.mean(output[series][experiment]['avg_idleness'], axis=0)
                for experiment in output[series]
            }
            for series in output
        }
    },
    'communication_requests': {
        'title': 'Communication Requests by Series',
        'ylabel': 'Communication Requests',
        'xlabel': 'Series',
        'y': {
            series: [np.mean(output[series][experiment]['communication']['requests_made']) for experiment in output[series]]
                for series in output
        }
    },
    'communication_requests_time': {
        'title': 'Cumulative Communication Requests vs. Time',
        'ylabel': 'Communication Requests',
        'xlabel': 'Time Steps',
        'y': {
            series: {
                experiment: np.cumsum(np.mean(output[series][experiment]['communication']['requests_made'], axis=0))
                for experiment in output[series]
            }
            for series in output
        }
    }
}

# Miscelaneous statistics.
timesteps = [np.max([np.max(output[series][experiment]['deltaSteps'].shape[1]) for experiment in output[series]]) for series in output]

for category in categories:
    fig = go.Figure()
    x = categories[category].get('x', None)
    y_data = categories[category]['y']
    show_legend = True

    # Detect nested dict: {series: {experiment: array}} — data with a time dimension.
    is_time_series = (
        isinstance(y_data, dict)
        and len(y_data) > 0
        and isinstance(next(iter(y_data.values())), dict)
    )

    if is_time_series:
        for series, tests in y_data.items():
            for test_name, values in tests.items():
                x_local = np.arange(1, len(values) + 1)
                fig.add_trace(go.Scatter(
                    x=x_local,
                    y=values,
                    mode='lines',
                    name=f"{series} | {test_name}",
                    hovertemplate=f'{series} | {test_name}' + ': %{y:.4f}<extra></extra>',
                    line={
                        "color": 'red' if "no-prediction" in series else 'blue',
                    }
                ))
    elif isinstance(y_data, dict):
        for series, values in y_data.items():
            x_local = x
            if x_local is None:
                x_local = np.linspace(0, 1, len(values))
            if len(values) == 1:
                # Plot a bar instead of a point if there's only one value, to make it more visible.
                fig.add_trace(go.Bar(
                    x=[series],
                    y=values,
                    name=series,
                    hovertemplate=f'{series}' + ': %{y:.4f}<extra></extra>',
                ))
                show_legend = False  # Legend is not needed if we're labeling bars directly.
            else:
                fig.add_trace(go.Scatter(
                    x=x_local,
                    y=values,
                    mode='lines+markers',
                    name=series,
                    hovertemplate='%{x:.2f}: %{y:.4f}<extra></extra>'
                ))
    else:
        values = y_data
        x_local = x
        if x_local is None:
            x_local = np.linspace(0, 1, len(values))
        fig.add_trace(go.Scatter(
            x=x_local,
            y=values,
            mode='lines+markers',
            hovertemplate='%{x:.2f}: %{y:.4f}<extra></extra>'
        ))
    
    title = categories[category]['title']
    xlabel = 'Scenario' if "xlabel" not in categories[category] else categories[category]['xlabel']
    ylabel = f"Timesteps out of {timesteps}" if "ylabel" not in categories[category] else categories[category]['ylabel']
    
    fig.update_layout(
        # title=f"{title}: avg. over {args['eval_episodes']['value']} episodes, seed {args['seed']['value']}",
        title=f"{title}",
        xaxis_title=xlabel,
        yaxis_title=ylabel,
        hovermode='closest',
        showlegend=show_legend,
        autosize=True,
        height=600,
        margin=dict(l=60, r=20, t=60, b=120),
        legend=dict(
            orientation='h',
            yanchor='top',
            y=-0.2,
            xanchor='center',
            x=0.5
        ),
    )
    
    fig.show()
